# 03 · API y página web

Levanta el servidor que usa la página web:
- **Carga todas las funciones** de los notebooks `02_ejecucion` (sistema completo con YOLO) y `04_opencv` (detector clásico de OpenCV), sin ejecutar sus demos.
- En la página, el desplegable **Método de reconocimiento** elige entre el **modelo YOLO11-seg** y **OpenCV**. La calibración y el cálculo de X, Y, Z son los mismos para los dos.
- Define la **API** (FastAPI): calibración, detección por imagen y detección en vivo por WebSocket.
- Sirve la **página web** en la raíz `/`.
- La publica con un **túnel Cloudflare** (`https://…trycloudflare.com`). Abrís ese link y listo: al ser HTTPS, el navegador permite usar las cámaras.

> En Colab: *Entorno de ejecución → Cambiar tipo → GPU T4*. La URL cambia cada vez que se ejecuta.
> Antes, subí la carpeta del proyecto a `MyDrive/tomates3d/`, con `notebooks/` adentro.

In [1]:
# ---------------------------------------------------------------- configuración e instalación
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

PORT = 8000
if EN_COLAB:
    from google.colab import drive
    if not os.path.isdir("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    NB02 = "/content/drive/MyDrive/tomates3d/notebooks/02_ejecucion.ipynb"
    NB04 = "/content/drive/MyDrive/tomates3d/notebooks/04_opencv.ipynb"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn[standard]", "python-multipart"], check=True)
    if not os.path.isfile("/content/cloudflared"):
        subprocess.run(["wget", "-q", "-O", "/content/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        os.chmod("/content/cloudflared", 0o755)
else:
    NB02 = "02_ejecucion.ipynb"  # ejecutando desde la carpeta notebooks/ en la PC
    NB04 = "04_opencv.ipynb"
for _nb in (NB02, NB04):
    assert os.path.isfile(_nb), f"No encuentro {_nb}"
print("Funciones desde:", NB02, "y", NB04)

Mounted at /content/drive
Funciones desde: /content/drive/MyDrive/tomates3d/notebooks/02_ejecucion.ipynb y /content/drive/MyDrive/tomates3d/notebooks/04_opencv.ipynb


## Cargar las funciones de los notebooks 02 y 04
`DEMO = False` hace que los notebooks solo definan funciones, sin correr sus demostraciones. El 02 aporta calibración, SGBM, YOLO y localización 3D; el 04, el detector OpenCV.

In [2]:
DEMO = False
%run -i "$NB02"
%run -i "$NB04"

Colab: True | proyecto: /content/drive/MyDrive/tomates3d
pesos: /content/drive/MyDrive/tomates3d/weights/best.pt (existe)
calibración: /content/drive/MyDrive/tomates3d/stereo_calib.yml (existe)
Funciones del notebook 02 cargadas: True
OpenCVTomatoDetector definido.


## API (FastAPI)

| Método | Ruta | Descripción |
|---|---|---|
| GET | `/` | Página web |
| GET | `/health` | Estado del modelo y de la calibración |
| POST | `/calibration/start` | JSON `{cols, rows, square_mm}` (esquinas internas del tablero) |
| POST | `/calibration/frame` | multipart `left`, `right` → si encontró el tablero, vista previa y cobertura |
| DELETE | `/calibration/frame/{i}` | Descarta un par |
| POST | `/calibration/compute` | Calibra y guarda; devuelve RMS, línea base, focal y error por par |
| GET / PUT | `/calibration` | Ver (`?format=yml` descarga) / subir un yml |
| POST | `/calibration/verify` | Par rectificado con líneas epipolares; con `x`, `y` devuelve Z en ese punto |
| POST | `/detect` | multipart `left`, `right`, `conf`, `images`, `method` (`modelo` u `opencv`) → tomates (maduros y semimaduros) con X, Y, Z |
| WS | `/ws/stream` | Binario: `uint32 BE len(left)` + JPEG izq + JPEG der → JSON por par |

In [3]:
import asyncio
import base64
import json
import struct
import tempfile

from fastapi import FastAPI, File, Form, HTTPException, Query, UploadFile, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel, Field

import traceback

INDEX_HTML = os.path.abspath("index.html")  # lo escribe la celda siguiente
ERRORES_LOG = os.path.abspath("errores_api.log")  # se muestra en la celda "Estado"


def registrar_error(origen: str, e: Exception) -> str:
    """Guarda el traceback completo en errores_api.log y devuelve un resumen para la página."""
    with open(ERRORES_LOG, "a", encoding="utf-8") as fh:
        fh.write(f"\n===== {time.strftime('%H:%M:%S')} {origen}\n{traceback.format_exc()}")
    return f"Error del servidor ({origen}): {type(e).__name__}: {e}"


METODOS = {"modelo": "Modelo YOLO11-seg", "opencv": "OpenCV (HSV + watershed)"}


class Estado:
    calib: StereoCalibration | None = None
    session: CalibrationSession | None = None
    detector = None                                # YOLO: se carga en segundo plano
    detector_error: str | None = None
    detector_opencv = OpenCVTomatoDetector()       # notebook 04: no necesita pesos
    _lock = threading.Lock()

    def get_detector(self, metodo: str = "modelo"):
        if metodo == "opencv":
            return self.detector_opencv
        if metodo != "modelo":
            raise HTTPException(400, f"Método desconocido '{metodo}'. Opciones: {', '.join(METODOS)}.")
        with self._lock:
            if self.detector is None and self.detector_error is None:
                try:
                    self.detector = TomatoDetector()
                except Exception as e:  # noqa: BLE001 - se informa en /health
                    self.detector_error = f"{type(e).__name__}: {e}"
        if self.detector is None:
            raise HTTPException(503, f"Modelo no disponible: {self.detector_error}")
        return self.detector


estado = Estado()
if os.path.isfile(CALIB_PATH):
    estado.calib = StereoCalibration.load(CALIB_PATH)
    print("Calibración cargada:", estado.calib.summary())


def _cargar_detector_silencioso():
    try:
        estado.get_detector()
    except HTTPException:
        pass


# El modelo se carga en segundo plano para que la API responda enseguida.
threading.Thread(target=_cargar_detector_silencioso, daemon=True).start()


app = FastAPI(title="Tomates 3D", version="2.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])


@app.exception_handler(Exception)
async def error_inesperado(request, e: Exception):
    return JSONResponse(status_code=500, content={"detail": registrar_error(request.url.path, e)})


def decode_image(data: bytes, name: str) -> np.ndarray:
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise HTTPException(400, f"No se pudo decodificar la imagen '{name}'.")
    return img


def b64jpg(img: np.ndarray, quality: int = 80, max_w: int | None = None) -> str:
    if max_w and img.shape[1] > max_w:
        img = cv2.resize(img, None, fx=max_w / img.shape[1], fy=max_w / img.shape[1], interpolation=cv2.INTER_AREA)
    _, buf = cv2.imencode(".jpg", img, [cv2.IMWRITE_JPEG_QUALITY, quality])
    return "data:image/jpeg;base64," + base64.b64encode(buf).decode()


def require_calib() -> StereoCalibration:
    if estado.calib is None:
        raise HTTPException(409, "No hay calibración activa. Calibrá las cámaras desde la pestaña 'Calibrar'.")
    return estado.calib


def require_session() -> CalibrationSession:
    if estado.session is None:
        raise HTTPException(409, "No hay sesión de calibración. Llamá primero a /calibration/start.")
    return estado.session


def procesar_para_api(left: np.ndarray, right: np.ndarray, conf: float, images: bool,
                      metodo: str = "modelo") -> dict:
    calib = require_calib()
    detector = estado.get_detector(metodo)
    try:
        out = procesar_par(left, right, calib, detector, conf=conf, imagenes=images)
    except ValueError as e:
        raise HTTPException(400, str(e)) from e
    out["method"] = metodo
    if images:
        out["annotated_jpg"] = b64jpg(out.pop("annotated"), max_w=1280)
        out["depth_jpg"] = b64jpg(out.pop("depth"), max_w=1280)
    return out


# ---------------------------------------------------------------- página y estado
@app.get("/")
def index():
    return FileResponse(INDEX_HTML, media_type="text/html")


@app.get("/health")
def health():
    det = estado.detector
    return {
        "status": "ok",
        "model_loaded": det is not None,
        "model": det.info() if det is not None else None,
        "model_error": estado.detector_error,
        "methods": METODOS,
        "calibrated": estado.calib is not None,
        "calibration": estado.calib.summary() if estado.calib else None,
    }


# ---------------------------------------------------------------- calibración
class StartRequest(BaseModel):
    cols: int = Field(9, ge=3, description="Esquinas internas horizontales")
    rows: int = Field(6, ge=3, description="Esquinas internas verticales")
    square_mm: float = Field(25.0, gt=0, description="Lado del cuadro en mm")


@app.post("/calibration/start")
def calibration_start(req: StartRequest):
    estado.session = CalibrationSession((req.cols, req.rows), req.square_mm)
    return {"ok": True, "pattern": [req.cols, req.rows], "square_mm": req.square_mm, "min_pairs": MIN_PAIRS}


@app.post("/calibration/frame")
def calibration_frame(left: UploadFile = File(...), right: UploadFile = File(...)):
    session = require_session()
    try:
        res = session.add_pair(decode_image(left.file.read(), "left"), decode_image(right.file.read(), "right"))
    except ValueError as e:
        raise HTTPException(400, str(e)) from e
    res["preview"] = b64jpg(res["preview"], quality=70)
    res["n_valid"] = len(session.pairs)
    res["min_pairs"] = MIN_PAIRS
    return res


@app.delete("/calibration/frame/{idx}")
def calibration_delete(idx: int):
    session = require_session()
    if not session.remove_pair(idx):
        raise HTTPException(404, f"No existe el par {idx}.")
    return {"ok": True, "n_valid": len(session.pairs)}


@app.post("/calibration/compute")
def calibration_compute():
    session = require_session()
    try:
        calib, report = session.compute()
    except ValueError as e:
        raise HTTPException(400, str(e)) from e
    except cv2.error as e:
        raise HTTPException(500, f"OpenCV falló al calibrar: {e}") from e
    calib.save(CALIB_PATH)
    estado.calib = calib
    report["saved_path"] = CALIB_PATH
    return report


@app.get("/calibration")
def calibration_get(format: str = Query("json", pattern="^(json|yml)$")):
    calib = require_calib()
    if format == "yml":
        return FileResponse(CALIB_PATH, filename="stereo_calib.yml", media_type="application/x-yaml")
    return calib.summary()


class EscalaRequest(BaseModel):
    factor: float | None = Field(None, gt=0.5, lt=1.5, description="Multiplica la línea base")
    baseline_cm: float | None = Field(None, gt=1, lt=100, description="Línea base medida con regla")


@app.post("/calibration/scale")
def calibration_scale(req: EscalaRequest):
    """Corrige la escala de la calibración activa (línea base) y la guarda."""
    calib = require_calib()
    if req.factor is None and req.baseline_cm is None:
        raise HTTPException(400, "Indicá factor o baseline_cm.")
    factor = req.factor if req.factor is not None else req.baseline_cm / (calib.baseline * 100)
    try:
        nueva = escalar_calibracion(calib, factor)
    except ValueError as e:
        raise HTTPException(400, str(e)) from e
    nueva.save(CALIB_PATH)
    estado.calib = nueva
    return {**nueva.summary(), "factor": factor}


@app.put("/calibration")
def calibration_put(file: UploadFile = File(...)):
    with tempfile.NamedTemporaryFile(suffix=".yml", delete=False) as tmp:
        tmp.write(file.file.read())
    try:
        calib = StereoCalibration.load(tmp.name)
    except Exception as e:  # noqa: BLE001
        raise HTTPException(400, f"Calibración inválida: {e}") from e
    finally:
        os.unlink(tmp.name)
    calib.save(CALIB_PATH)
    estado.calib = calib
    return calib.summary()


@app.post("/calibration/verify")
def calibration_verify(left: UploadFile = File(...), right: UploadFile = File(...),
                       x: float | None = Form(None), y: float | None = Form(None)):
    calib = require_calib()
    try:
        rl, rr = calib.rectify(decode_image(left.file.read(), "left"), decode_image(right.file.read(), "right"))
    except ValueError as e:
        raise HTTPException(400, str(e)) from e
    out = {"image": b64jpg(epipolar_preview(rl, rr), max_w=1600), "image_size": list(calib.image_size)}
    if x is not None and y is not None:
        disp = compute_disparity(rl, rr, max_disparity=calib.f * calib.baseline / MIN_DEPTH_M)
        z, d = depth_at(disp, x, y, calib.f, calib.baseline)
        out.update({"x": x, "y": y, "z_m": z, "disparity_px": d})
    return out


# ---------------------------------------------------------------- detección
@app.post("/detect")
def detect(left: UploadFile = File(...), right: UploadFile = File(...),
           conf: float = Form(0.35), images: bool = Form(True), method: str = Form("modelo")):
    return procesar_para_api(decode_image(left.file.read(), "left"),
                             decode_image(right.file.read(), "right"), conf, images, method)


@app.websocket("/ws/stream")
async def stream(ws: WebSocket):
    """Texto JSON opcional {"conf": 0.35, "method": "modelo"}; binario uint32 BE len(left) + JPEG izq + JPEG der.
    El cliente espera cada respuesta antes de enviar el siguiente par, así no se acumula atraso."""
    await ws.accept()
    params = {"conf": 0.35, "method": "modelo"}
    try:
        while True:
            msg = await ws.receive()
            if msg["type"] == "websocket.disconnect":
                break
            if msg.get("text") is not None:
                try:
                    nuevos = json.loads(msg["text"])
                    if nuevos.get("method", params["method"]) not in METODOS:
                        await ws.send_json({"type": "error", "detail": f"Método desconocido: {nuevos['method']}"})
                        continue
                    params.update(nuevos)
                    await ws.send_json({"type": "params", **params})
                except json.JSONDecodeError:
                    await ws.send_json({"type": "error", "detail": "JSON inválido"})
                continue
            data = msg.get("bytes") or b""
            try:
                (n,) = struct.unpack(">I", data[:4])
                l_img = decode_image(data[4:4 + n], "left")
                r_img = decode_image(data[4 + n:], "right")
                res = await asyncio.to_thread(procesar_para_api, l_img, r_img, float(params["conf"]), False,
                                        params["method"])
                await ws.send_json({"type": "result", **res})
            except HTTPException as e:
                await ws.send_json({"type": "error", "status": e.status_code, "detail": e.detail})
            except struct.error:
                await ws.send_json({"type": "error", "detail": "Mensaje binario mal formado"})
            except Exception as e:  # noqa: BLE001 - antes cerraba la conexión sin explicar por qué
                await ws.send_json({"type": "error", "detail": registrar_error("/ws/stream", e)})
    except WebSocketDisconnect:
        pass

print("API definida.")

Calibración cargada: {'image_size': [1280, 720], 'rms': 0.36798735644403574, 'baseline_cm': 8.138774676806273, 'focal_px': 1605.931300510672, 'created': '2026-11-11T22:53:10'}
API definida.


## Página web
Una sola página con tres pestañas: **Calibrar** (asistente con tablero), **Imagen** (subir o capturar un par) y **En vivo** (video de las cámaras con overlay y gráfico 3D). La celda escribe `index.html` y la API la sirve en `/`. El desplegable **Método de reconocimiento** (panel izquierdo) vale para las pestañas Imagen y En vivo.


In [4]:
%%writefile index.html
<!doctype html>
<html lang="es">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Tomates 3D</title>
  <style>
:root {
  --bg: #f6f5f2;
  --panel: #ffffff;
  --text: #1e1f1c;
  --muted: #6b6d66;
  --line: #e2e0da;
  --accent: #d63a2b;
  --accent-ink: #ffffff;
  --ok: #2e8b4a;
  --warn: #c98a12;
  --bad: #c0392b;
  --radius: 10px;
  font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif;
}
@media (prefers-color-scheme: dark) {
  :root {
    --bg: #151614;
    --panel: #1f201d;
    --text: #ecebe6;
    --muted: #9a9b94;
    --line: #34352f;
    --accent: #ef5a43;
  }
}
* { box-sizing: border-box; }
body { margin: 0; background: var(--bg); color: var(--text); font-size: 14px; }
h2, h3 { margin: 0 0 12px; font-size: 15px; font-weight: 600; }
.small { font-size: 12px; }
.muted { color: var(--muted); }
.mono { font-family: ui-monospace, Consolas, monospace; }
.big { font-size: 20px; font-weight: 600; font-variant-numeric: tabular-nums; }
.hidden { display: none !important; }

.topbar {
  display: flex; gap: 16px; align-items: center; flex-wrap: wrap;
  padding: 10px 20px; background: var(--panel); border-bottom: 1px solid var(--line);
  position: sticky; top: 0; z-index: 5;
}
.brand { font-weight: 700; font-size: 16px; display: flex; align-items: center; gap: 8px; }
.brand small { font-weight: 400; color: var(--muted); }
.brand .dot { width: 14px; height: 14px; border-radius: 50%; background: var(--accent); box-shadow: inset -3px -3px 0 rgba(0,0,0,.15); }
.conn { display: flex; gap: 6px; flex: 1; min-width: 260px; max-width: 520px; }
.conn input { flex: 1; }
.chips { display: flex; gap: 6px; }
.chip { padding: 3px 10px; border-radius: 99px; font-size: 12px; border: 1px solid var(--line); color: var(--muted); }
.chip.on { background: color-mix(in srgb, var(--ok) 15%, transparent); color: var(--ok); border-color: color-mix(in srgb, var(--ok) 40%, transparent); }
.chip.warn { background: color-mix(in srgb, var(--warn) 15%, transparent); color: var(--warn); border-color: color-mix(in srgb, var(--warn) 40%, transparent); }

.layout { display: grid; grid-template-columns: 300px 1fr; gap: 16px; padding: 16px 20px; align-items: start; }
@media (max-width: 900px) { .layout { grid-template-columns: 1fr; padding: 12px 16px; } }
.panel { background: var(--panel); border: 1px solid var(--line); border-radius: var(--radius); padding: 14px; margin: 0 0 14px; }
.cams { position: sticky; top: 70px; }
.cams label { display: block; margin-bottom: 8px; }
.cams select { width: 100%; }
.videos { display: grid; grid-template-columns: 1fr 1fr; gap: 6px; margin-top: 10px; }
.videos figure { margin: 0; position: relative; }
.videos video { width: 100%; aspect-ratio: 16/9; background: #000; border-radius: 6px; object-fit: contain; }
.videos figcaption { position: absolute; top: 4px; left: 6px; color: #fff; font-size: 11px; font-weight: 700; text-shadow: 0 1px 2px #000; }

label { color: var(--muted); font-size: 12px; display: inline-flex; flex-direction: column; gap: 4px; }
label.check { flex-direction: row; align-items: center; gap: 6px; color: var(--text); font-size: 13px; }
input, select {
  font: inherit; color: var(--text); background: var(--bg);
  border: 1px solid var(--line); border-radius: 6px; padding: 6px 8px;
}
button, .file {
  font: inherit; font-weight: 600; cursor: pointer; border-radius: 6px; padding: 7px 14px;
  border: 1px solid var(--accent); background: var(--accent); color: var(--accent-ink);
  transition: transform .1s ease, opacity .15s ease;
}
button:active { transform: scale(.97); }
button:disabled { opacity: .45; cursor: not-allowed; }
button.ghost, .file.ghost, .file { background: transparent; color: var(--text); border-color: var(--line); }
.file { flex-direction: row; align-items: center; justify-content: center; }
.file input { display: none; }
.file.has-file { border-color: var(--ok); color: var(--ok); }
.row { display: flex; gap: 10px; align-items: end; flex-wrap: wrap; margin: 8px 0; }
.grid3 { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; align-items: end; margin-bottom: 8px; }
@media (max-width: 600px) { .grid3 { grid-template-columns: 1fr; } }

.tabs { display: flex; gap: 4px; margin-bottom: 14px; border-bottom: 1px solid var(--line); }
.tabs button { background: none; border: none; color: var(--muted); border-bottom: 2px solid transparent; border-radius: 0; padding: 8px 14px; }
.tabs button.active { color: var(--text); border-bottom-color: var(--accent); }
.tab { display: none; }
.tab.active { display: block; }

.badge { font-size: 12px; padding: 2px 8px; border-radius: 99px; background: var(--bg); border: 1px solid var(--line); margin-left: 6px; font-variant-numeric: tabular-nums; }
.badge.ok { color: var(--ok); border-color: var(--ok); }
.tips { margin: 6px 0 10px; padding-left: 18px; color: var(--muted); }
.calib-split { display: grid; grid-template-columns: 320px 1fr; gap: 12px; align-items: start; }
@media (max-width: 700px) { .calib-split { grid-template-columns: 1fr; } }
#coverage { width: 100%; max-width: 320px; background: #000; border-radius: 6px; }
.preview { width: 100%; border-radius: 6px; }
.thumbs { display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 8px; margin-top: 10px; }
.thumb { position: relative; border: 2px solid var(--ok); border-radius: 6px; overflow: hidden; }
.thumb.bad { border-color: var(--bad); opacity: .6; }
.thumb img { width: 100%; display: block; }
.thumb span { position: absolute; left: 4px; bottom: 4px; font-size: 11px; background: rgba(0,0,0,.65); color: #fff; padding: 1px 5px; border-radius: 4px; }
.thumb button { position: absolute; top: 3px; right: 3px; padding: 0 7px; font-size: 13px; line-height: 20px; background: rgba(0,0,0,.6); border: none; }

.result { margin-top: 10px; }
.kpis { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 10px; }
.kpi { border: 1px solid var(--line); border-radius: 8px; padding: 10px; }
.kpi b { display: block; font-size: 20px; font-variant-numeric: tabular-nums; }
.kpi.good b { color: var(--ok); } .kpi.mid b { color: var(--warn); } .kpi.bad b { color: var(--bad); }
.bars { margin-top: 12px; display: grid; gap: 3px; }
.bar { display: grid; grid-template-columns: 40px 1fr 60px 28px; gap: 8px; align-items: center; font-size: 12px; font-variant-numeric: tabular-nums; }
.bar i { display: block; height: 8px; border-radius: 4px; background: var(--ok); }
.bar i.hi { background: var(--bad); }
.bar button { padding: 0 6px; font-size: 12px; background: transparent; color: var(--muted); border-color: var(--line); }
.verify { width: 100%; margin-top: 10px; border-radius: 6px; cursor: crosshair; }

.gate { padding: 12px 14px; border-radius: var(--radius); margin-bottom: 14px; background: color-mix(in srgb, var(--warn) 14%, transparent); color: var(--warn); font-weight: 600; }
.results-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 14px; }
@media (max-width: 1100px) { .results-grid { grid-template-columns: 1fr; } }
.results-grid figure { margin: 0 0 14px; }
.results-grid figcaption { font-weight: 600; margin-bottom: 8px; }
.results-grid img { width: 100%; border-radius: 6px; background: var(--bg); min-height: 60px; }
.plot { width: 100%; height: 380px; }
.stage { position: relative; padding: 0; overflow: hidden; background: #000; align-self: start; }
.stage video { width: 100%; display: block; }
.stage canvas { position: absolute; top: 0; left: 0; width: 100%; height: auto; }  /* misma proporción que el video */

table { width: 100%; border-collapse: collapse; font-size: 13px; font-variant-numeric: tabular-nums; }
th, td { text-align: right; padding: 6px 8px; border-bottom: 1px solid var(--line); }
th:first-child, td:first-child, th:nth-child(2), td:nth-child(2) { text-align: left; }
th { color: var(--muted); font-weight: 500; }
.swatch { display: inline-block; width: 10px; height: 10px; border-radius: 50%; margin-right: 6px; vertical-align: middle; }

.toast {
  position: fixed; bottom: 20px; left: 50%; transform: translateX(-50%);
  background: var(--text); color: var(--bg); padding: 10px 16px; border-radius: 8px;
  max-width: min(90vw, 560px); z-index: 20; box-shadow: 0 6px 24px rgba(0,0,0,.25);
}
.toast.error { background: var(--bad); color: #fff; }
</style>
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
</head>
<body>
<header class="topbar">
  <div class="brand"><span class="dot"></span> Tomates 3D <small>tomates maduros y semimaduros · estéreo</small></div>
  <div class="chips">
    <span id="chipApi" class="chip off">API</span>
    <span id="chipModel" class="chip off">Modelo</span>
    <span id="chipCalib" class="chip off">Calibración</span>
  </div>
</header>

<main class="layout">
  <!-- Cámaras: compartidas por todas las pestañas -->
  <aside class="panel cams">
    <h2>Cámaras</h2>
    <label>Izquierda <select id="camLeft"></select></label>
    <label>Derecha <select id="camRight"></select></label>
    <label>Resolución
      <select id="camRes">
        <option value="640x480">640 × 480</option>
        <option value="1280x720" selected>1280 × 720</option>
        <option value="1920x1080">1920 × 1080</option>
      </select>
    </label>
    <div class="row">
      <button id="btnCams">Abrir cámaras</button>
      <button id="btnSwap" class="ghost" title="Intercambiar izquierda/derecha">⇄</button>
    </div>
    <div class="videos">
      <figure><video id="vidLeft" autoplay playsinline muted></video><figcaption>L</figcaption></figure>
      <figure><video id="vidRight" autoplay playsinline muted></video><figcaption>R</figcaption></figure>
    </div>
    <p id="camInfo" class="muted small"></p>

    <h2 style="margin-top: 18px">Reconocimiento</h2>
    <label>Método de reconocimiento
      <select id="metodo">
        <option value="modelo">Modelo YOLO11-seg (entrenado)</option>
        <option value="opencv">OpenCV (color HSV + watershed)</option>
      </select>
    </label>
    <p id="metodoInfo" class="muted small"></p>
  </aside>

  <section class="content">
    <nav class="tabs">
      <button data-tab="calib" class="active">1 · Calibrar</button>
      <button data-tab="image">2 · Imagen</button>
      <button data-tab="live">3 · En vivo</button>
    </nav>

    <!-- ===================== CALIBRAR ===================== -->
    <div id="tab-calib" class="tab active">
      <div class="steps">
        <div class="panel">
          <h3>Paso 1 · Tablero</h3>
          <div class="grid3">
            <label>Esquinas internas (ancho) <input id="bCols" type="number" value="9" min="3"></label>
            <label>Esquinas internas (alto) <input id="bRows" type="number" value="6" min="3"></label>
            <label>Lado del cuadro (mm) <input id="bSquare" type="number" value="25" step="0.1" min="1"></label>
          </div>
          <div class="row">
            <button id="btnStartCalib">Iniciar sesión de calibración</button>
            <button id="btnBoard" class="ghost">Generar tablero imprimible</button>
          </div>
          <p class="muted small">Imprimí el tablero sin escalar, pegalo sobre algo rígido y <b>medí el cuadro con una regla</b>: ese valor es el que importa.</p>
        </div>

        <div class="panel">
          <h3>Paso 2 · Capturar pares <span id="pairCount" class="badge">0 / 12</span></h3>
          <div class="row">
            <button id="btnCapture" disabled>Capturar par</button>
            <label class="check"><input id="autoCapture" type="checkbox" disabled> Captura automática (cada 2 s)</label>
          </div>
          <ul class="tips small">
            <li>Mostrá el tablero completo en <b>ambas</b> cámaras.</li>
            <li>Variá distancia (cerca/lejos), inclinación (±30°) y posición: cubrí los bordes y esquinas.</li>
            <li>Mantené el tablero quieto en cada captura (las webcams no están sincronizadas).</li>
          </ul>
          <div class="calib-split">
            <div>
              <p class="small muted">Cobertura del tablero en la cámara izquierda</p>
              <canvas id="coverage" width="320" height="180"></canvas>
            </div>
            <img id="lastPreview" class="preview" alt="">
          </div>
          <div id="thumbs" class="thumbs"></div>
        </div>

        <div class="panel">
          <h3>Paso 3 · Calcular</h3>
          <button id="btnCompute" disabled>Calcular calibración</button>
          <div id="calibResult" class="result hidden"></div>
        </div>

        <div class="panel">
          <h3>Paso 4 · Verificar</h3>
          <p class="small muted">Las líneas horizontales deben pasar por los mismos puntos en ambas imágenes. Hacé clic sobre la imagen izquierda para medir Z y compararla con una cinta métrica.</p>
          <div class="row">
            <button id="btnVerify" disabled>Capturar y rectificar</button>
            <span id="verifyZ" class="big"></span>
          </div>
          <img id="verifyImg" class="verify hidden" alt="Par rectificado">
        </div>

        <div class="panel">
          <h3>Paso 5 · Ajuste de escala (opcional)</h3>
          <p class="small muted">Si las distancias salen todas un poco grandes o chicas en la misma proporción, el error está en la línea base.
            Medí el tomate a 2–5 distancias distintas con cinta (desde la lente izquierda hasta el <b>centro</b> del tomate) y anotá
            lo que marca la columna <b>Z centro máscaras</b> de la pestaña En vivo. El factor <i>k</i> se calcula por mínimos cuadrados.</p>
          <div class="grid3">
            <label>Z medida por el sistema (cm) <input id="escZm" type="number" step="0.1" min="1"></label>
            <label>Z real con cinta (cm) <input id="escZr" type="number" step="0.1" min="1"></label>
            <div class="row" style="margin:0">
              <button id="btnEscUltima" class="ghost" title="Copia la Z del primer tomate del modo En vivo">Usar última Z en vivo</button>
              <button id="btnEscAgregar">Agregar</button>
            </div>
          </div>
          <div id="escTabla"></div>
          <div class="row">
            <button id="btnEscAplicar" disabled>Aplicar factor</button>
            <span id="escInfo" class="muted small"></span>
          </div>
          <details class="small muted"><summary>…o fijar la línea base medida con regla</summary>
            <div class="row">
              <label>Línea base (cm, centro a centro de las lentes) <input id="escBase" type="number" step="0.01" min="1"></label>
              <button id="btnEscBase" class="ghost">Usar esta línea base</button>
            </div>
          </details>
        </div>

        <div class="panel">
          <h3>Guardar / cargar</h3>
          <div class="row">
            <button id="btnDownloadCalib" class="ghost" disabled>Descargar .yml</button>
            <label class="file ghost">Subir .yml <input id="uploadCalib" type="file" accept=".yml,.yaml,.xml"></label>
          </div>
          <p class="small muted">La calibración también queda guardada en el servidor (Google Drive en Colab) y se recarga sola.</p>
        </div>
      </div>
    </div>

    <!-- ===================== IMAGEN ===================== -->
    <div id="tab-image" class="tab">
      <div class="panel">
        <h3>Par estéreo</h3>
        <div class="grid3">
          <label class="file">Imagen izquierda <input id="fileLeft" type="file" accept="image/*"></label>
          <label class="file">Imagen derecha <input id="fileRight" type="file" accept="image/*"></label>
          <button id="btnSnap" class="ghost">…o capturar de las cámaras</button>
        </div>
        <div class="row">
          <label>Confianza <input id="imgConf" type="number" value="0.35" min="0.05" max="0.95" step="0.05"></label>
          <button id="btnDetect">Detectar</button>
          <span id="imgStatus" class="muted small"></span>
        </div>
      </div>
      <div class="gate hidden" data-gate>Primero calibrá las cámaras (pestaña 1) o subí un .yml.</div>
      <div class="results-grid">
        <figure class="panel"><figcaption>Detección (izquierda rectificada)</figcaption><img id="imgAnnotated" alt=""></figure>
        <figure class="panel"><figcaption>Profundidad</figcaption><img id="imgDepth" alt=""></figure>
      </div>
      <div class="results-grid">
        <div class="panel"><h3>Coordenadas</h3><div id="imgTable"></div></div>
        <div class="panel"><h3>Vista 3D</h3><div id="imgPlot" class="plot"></div></div>
      </div>
    </div>

    <!-- ===================== EN VIVO ===================== -->
    <div id="tab-live" class="tab">
      <div class="panel row">
        <label>Confianza <input id="liveConf" type="number" value="0.35" min="0.05" max="0.95" step="0.05"></label>
        <label>Calidad JPEG <input id="liveQuality" type="number" value="0.7" min="0.3" max="0.95" step="0.05"></label>
        <button id="btnLive">Iniciar</button>
        <span id="liveStats" class="mono small"></span>
      </div>
      <div class="gate hidden" data-gate>Primero calibrá las cámaras (pestaña 1) o subí un .yml.</div>
      <div class="results-grid">
        <div class="panel stage">
          <video id="liveVideo" autoplay playsinline muted></video>
          <canvas id="liveOverlay"></canvas>
        </div>
        <div class="panel"><h3>Vista 3D</h3><div id="livePlot" class="plot"></div></div>
      </div>
      <div class="panel"><h3>Coordenadas</h3><div id="liveTable"></div></div>
    </div>
  </section>
</main>

<div id="toast" class="toast hidden"></div>

<script>
// Estado compartido: conexión con la API y manejo de las dos cámaras.
const $ = (sel) => document.querySelector(sel);

const App = {
  api: "",
  health: null,
  streams: { left: null, right: null },
};

// ------------------------------------------------------------- utilidades
function toast(msg, isError = false, ms = 3500) {
  const t = $("#toast");
  t.textContent = msg;
  t.classList.toggle("error", isError);
  t.classList.remove("hidden");
  clearTimeout(toast._t);
  toast._t = setTimeout(() => t.classList.add("hidden"), ms);
}

async function api(path, opts = {}) {
  if (!App.api) throw new Error("API no disponible.");
  const res = await fetch(App.api + path, opts);
  const ct = res.headers.get("content-type") || "";
  const body = ct.includes("json") ? await res.json() : await res.blob();
  if (!res.ok) {
    const detail = body && body.detail ? (typeof body.detail === "string" ? body.detail : JSON.stringify(body.detail)) : res.statusText;
    throw new Error(`${res.status}: ${detail}`);
  }
  return body;
}

function pairForm(left, right, extra = {}) {
  const fd = new FormData();
  fd.append("left", left, "left.jpg");
  fd.append("right", right, "right.jpg");
  for (const [k, v] of Object.entries(extra)) if (v !== undefined && v !== null) fd.append(k, v);
  return fd;
}

// ------------------------------------------------------------- conexión
async function refreshHealth() {
  try {
    App.health = await api("/health");
  } catch (e) {
    App.health = null;
  }
  const h = App.health;
  setChip("#chipApi", !!h, h ? "API" : "API sin conexión");
  if (metodo() === "opencv") setChip("#chipModel", !!h, "OpenCV (HSV)");
  else setChip("#chipModel", h && h.model_loaded, h && h.model_loaded
    ? (h.model && h.model.proxy ? "Modelo (sustituto COCO)" : "Modelo " + (h.model && h.model.gpu ? "(GPU)" : "(CPU)")) : (h && h.model_error ? "Modelo: error" : "Modelo…"),
    h && h.model && h.model.proxy);
  setChip("#chipCalib", h && h.calibrated, h && h.calibrated
    ? `Calib. B=${h.calibration.baseline_cm.toFixed(1)} cm` : "Sin calibrar");
  document.querySelectorAll("[data-gate]").forEach((g) => g.classList.toggle("hidden", !!(h && h.calibrated)));
  $("#btnVerify").disabled = !(h && h.calibrated);
  $("#btnDownloadCalib").disabled = !(h && h.calibrated);
  return h;
}

// ------------------------------------------------------------- método de reconocimiento
const metodo = () => $("#metodo").value;
const METODO_INFO = {
  modelo: "YOLO11-seg entrenado con Laboro Tomato: tomates maduros y semimaduros. Confianza = probabilidad del modelo.",
  opencv: "Segmentación por color rojo (HSV) + watershed para separar tomates que se tocan. Solo tomates rojos; " +
          "sensible a la luz y a objetos rojos. Confianza = redondez mínima (0–1).",
};

function onMetodo() {
  $("#metodoInfo").textContent = METODO_INFO[metodo()];
  try { localStorage.setItem("tomates.metodo", metodo()); } catch {}
  refreshHealth();
  sendParams();
}
$("#metodo").addEventListener("change", onMetodo);

function setChip(sel, on, text, warn = false) {
  const c = $(sel);
  c.textContent = text;
  c.className = "chip " + (on ? (warn ? "warn" : "on") : "off");
}


// ------------------------------------------------------------- cámaras
async function listCameras() {
  // Pedir permiso una vez para que enumerateDevices devuelva los nombres.
  try {
    const tmp = await navigator.mediaDevices.getUserMedia({ video: true });
    tmp.getTracks().forEach((t) => t.stop());
  } catch (e) {
    toast("Sin permiso para usar la cámara: " + e.message, true);
    return;
  }
  const devices = (await navigator.mediaDevices.enumerateDevices()).filter((d) => d.kind === "videoinput");
  const saved = loadCamPrefs();
  for (const [sel, pref, idx] of [["#camLeft", saved.left, 0], ["#camRight", saved.right, 1]]) {
    const s = $(sel);
    s.innerHTML = devices.map((d, i) => `<option value="${d.deviceId}">${d.label || "Cámara " + (i + 1)}</option>`).join("");
    if (pref && devices.some((d) => d.deviceId === pref)) s.value = pref;
    else if (devices[idx]) s.value = devices[idx].deviceId;
  }
  $("#camInfo").textContent = devices.length < 2
    ? "Se detectó menos de 2 cámaras. Conectá ambas webcams (idealmente en puertos USB distintos)."
    : `${devices.length} cámaras disponibles.`;
}

function loadCamPrefs() {
  try { return JSON.parse(localStorage.getItem("tomates.cams") || "{}"); } catch { return {}; }
}

async function openCameras() {
  stopCameras();
  const [w, h] = $("#camRes").value.split("x").map(Number);
  const left = $("#camLeft").value, right = $("#camRight").value;
  if (!left || !right || left === right) {
    toast("Elegí dos cámaras distintas para izquierda y derecha.", true);
    return false;
  }
  const constraints = (id) => ({ video: { deviceId: { exact: id }, width: { exact: w }, height: { exact: h } }, audio: false });
  try {
    App.streams.left = await navigator.mediaDevices.getUserMedia(constraints(left));
    App.streams.right = await navigator.mediaDevices.getUserMedia(constraints(right));
  } catch (e) {
    stopCameras();
    toast(`No se pudieron abrir las cámaras a ${w}×${h}: ${e.name}. Probá otra resolución o puertos USB distintos.`, true, 6000);
    return false;
  }
  $("#vidLeft").srcObject = App.streams.left;
  $("#vidRight").srcObject = App.streams.right;
  $("#liveVideo").srcObject = App.streams.left;
  try { localStorage.setItem("tomates.cams", JSON.stringify({ left, right, res: $("#camRes").value })); } catch {}
  await Promise.all([waitVideo($("#vidLeft")), waitVideo($("#vidRight"))]);
  $("#camInfo").textContent = `Abiertas a ${$("#vidLeft").videoWidth}×${$("#vidLeft").videoHeight}. ` +
    "Usá siempre la misma resolución para calibrar y detectar.";
  document.dispatchEvent(new Event("cameras-ready"));
  return true;
}

function stopCameras() {
  for (const k of ["left", "right"]) {
    if (App.streams[k]) App.streams[k].getTracks().forEach((t) => t.stop());
    App.streams[k] = null;
  }
}

function camerasOpen() {
  return !!(App.streams.left && App.streams.right);
}

function waitVideo(v) {
  return new Promise((resolve) => {
    if (v.readyState >= 2 && v.videoWidth) return resolve();
    v.addEventListener("loadeddata", () => resolve(), { once: true });
  });
}

const _grabCanvas = document.createElement("canvas");
function grabFrame(video, quality = 0.92) {
  _grabCanvas.width = video.videoWidth;
  _grabCanvas.height = video.videoHeight;
  _grabCanvas.getContext("2d").drawImage(video, 0, 0);
  return new Promise((resolve) => _grabCanvas.toBlob(resolve, "image/jpeg", quality));
}

// Captura casi simultánea de ambas cámaras (las webcams USB no tienen sincronismo por hardware).
async function grabPair(quality = 0.92) {
  if (!camerasOpen()) throw new Error("Abrí las cámaras primero.");
  const left = await grabFrame($("#vidLeft"), quality);
  const right = await grabFrame($("#vidRight"), quality);
  return { left, right };
}

$("#btnCams").addEventListener("click", openCameras);
$("#btnSwap").addEventListener("click", async () => {
  const l = $("#camLeft").value;
  $("#camLeft").value = $("#camRight").value;
  $("#camRight").value = l;
  if (camerasOpen()) await openCameras();
});

// ------------------------------------------------------------- pestañas
document.querySelectorAll(".tabs button").forEach((b) =>
  b.addEventListener("click", () => {
    document.querySelectorAll(".tabs button").forEach((x) => x.classList.toggle("active", x === b));
    document.querySelectorAll(".tab").forEach((t) => t.classList.toggle("active", t.id === "tab-" + b.dataset.tab));
    window.dispatchEvent(new Event("resize")); // para que Plotly reacomode los gráficos
  })
);

// ------------------------------------------------------------- inicio
(async function init() {
  const saved = loadCamPrefs();
  if (saved.res) $("#camRes").value = saved.res;
  App.api = window.location.origin;
  try { const m = localStorage.getItem("tomates.metodo"); if (m) $("#metodo").value = m; } catch {}
  $("#metodoInfo").textContent = METODO_INFO[metodo()];
  refreshHealth();
  await listCameras();
})();
setInterval(() => App.api && refreshHealth(), 10000);

// Asistente de calibración estéreo.
const Calib = {
  active: false,
  minPairs: 12,
  pairs: new Map(), // index -> { coverage, el }
  autoTimer: null,
  busy: false,
  verifyPair: null, // último par usado para verificar (para medir Z al hacer clic)
};

$("#btnStartCalib").addEventListener("click", async () => {
  const body = { cols: +$("#bCols").value, rows: +$("#bRows").value, square_mm: +$("#bSquare").value };
  try {
    const r = await api("/calibration/start", {
      method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(body),
    });
    Calib.active = true;
    Calib.minPairs = r.min_pairs;
    Calib.pairs.clear();
    $("#thumbs").innerHTML = "";
    $("#lastPreview").removeAttribute("src");
    $("#calibResult").classList.add("hidden");
    drawCoverage();
    updatePairCount();
    $("#btnCapture").disabled = false;
    $("#autoCapture").disabled = false;
    toast(`Sesión iniciada: tablero ${r.pattern[0]}×${r.pattern[1]}, cuadro ${r.square_mm} mm.`);
    if (!camerasOpen()) await openCameras();
  } catch (e) {
    toast(e.message, true);
  }
});

async function capturePair(silentFail = false) {
  if (Calib.busy) return;
  Calib.busy = true;
  try {
    const { left, right } = await grabPair(0.95);
    const r = await api("/calibration/frame", { method: "POST", body: pairForm(left, right) });
    $("#lastPreview").src = r.preview;
    if (r.index !== null) {
      addThumb(r.index, r.preview, r.coverage);
    } else if (!silentFail) {
      const which = [!r.found_left && "izquierda", !r.found_right && "derecha"].filter(Boolean).join(" y ");
      toast(`No se detectó el tablero en la cámara ${which}.`, true);
    }
    updatePairCount(r.n_valid);
  } catch (e) {
    toast(e.message, true);
    stopAuto();
  } finally {
    Calib.busy = false;
  }
}

function addThumb(index, src, coverage) {
  const el = document.createElement("div");
  el.className = "thumb";
  el.innerHTML = `<img src="${src}" alt=""><span>#${index}</span><button title="Descartar">×</button>`;
  el.querySelector("button").addEventListener("click", () => removePair(index));
  $("#thumbs").prepend(el);
  Calib.pairs.set(index, { coverage, el });
  drawCoverage();
}

async function removePair(index) {
  try {
    const r = await api(`/calibration/frame/${index}`, { method: "DELETE" });
    const p = Calib.pairs.get(index);
    if (p) p.el.remove();
    Calib.pairs.delete(index);
    drawCoverage();
    updatePairCount(r.n_valid);
  } catch (e) {
    toast(e.message, true);
  }
}

function updatePairCount(n = Calib.pairs.size) {
  const b = $("#pairCount");
  b.textContent = `${n} / ${Calib.minPairs}`;
  b.classList.toggle("ok", n >= Calib.minPairs);
  $("#btnCompute").disabled = n < Calib.minPairs;
}

function drawCoverage() {
  const c = $("#coverage"), ctx = c.getContext("2d");
  ctx.fillStyle = "#000";
  ctx.fillRect(0, 0, c.width, c.height);
  ctx.fillStyle = "rgba(80, 220, 120, 0.18)";
  ctx.strokeStyle = "rgba(80, 220, 120, 0.8)";
  for (const { coverage: [x, y, w, h] } of Calib.pairs.values()) {
    ctx.fillRect(x * c.width, y * c.height, w * c.width, h * c.height);
    ctx.strokeRect(x * c.width, y * c.height, w * c.width, h * c.height);
  }
  // Guía de 3x3: las zonas sin verde todavía no tienen muestras.
  ctx.strokeStyle = "rgba(255,255,255,0.25)";
  ctx.setLineDash([4, 4]);
  for (let i = 1; i < 3; i++) {
    ctx.beginPath(); ctx.moveTo((c.width * i) / 3, 0); ctx.lineTo((c.width * i) / 3, c.height); ctx.stroke();
    ctx.beginPath(); ctx.moveTo(0, (c.height * i) / 3); ctx.lineTo(c.width, (c.height * i) / 3); ctx.stroke();
  }
  ctx.setLineDash([]);
}

function stopAuto() {
  clearInterval(Calib.autoTimer);
  Calib.autoTimer = null;
  $("#autoCapture").checked = false;
}

$("#btnCapture").addEventListener("click", () => capturePair(false));
$("#autoCapture").addEventListener("change", (e) => {
  if (e.target.checked) Calib.autoTimer = setInterval(() => capturePair(true), 2000);
  else stopAuto();
});

// ------------------------------------------------------------- calcular
$("#btnCompute").addEventListener("click", async () => {
  stopAuto();
  const btn = $("#btnCompute");
  btn.disabled = true;
  btn.textContent = "Calculando…";
  try {
    const r = await api("/calibration/compute", { method: "POST" });
    renderCalibResult(r);
    await refreshHealth();
    toast(`Calibración lista: RMS ${r.rms.toFixed(3)} px, línea base ${r.baseline_cm.toFixed(2)} cm.`);
  } catch (e) {
    toast(e.message, true, 6000);
  } finally {
    btn.textContent = "Calcular calibración";
    updatePairCount();
  }
});

function renderCalibResult(r) {
  const grade = r.rms < 0.5 ? "good" : r.rms < 1 ? "mid" : "bad";
  const verdict = { good: "Buena", mid: "Aceptable", bad: "Recalibrar" }[grade];
  const maxErr = Math.max(...r.per_pair.map((p) => p.error), 1e-6);
  const threshold = Math.max(r.rms * 1.5, 0.5);
  const bars = [...r.per_pair].sort((a, b) => b.error - a.error).map((p) => `
    <div class="bar">
      <span>#${p.index}</span>
      <i class="${p.error > threshold ? "hi" : ""}" style="width:${(100 * p.error) / maxErr}%"></i>
      <span>${p.error.toFixed(3)} px</span>
      <button data-del="${p.index}" title="Descartar este par">×</button>
    </div>`).join("");
  const el = $("#calibResult");
  el.innerHTML = `
    <div class="kpis">
      <div class="kpi ${grade}"><span class="small muted">RMS estéreo</span><b>${r.rms.toFixed(3)} px</b><span class="small">${verdict}</span></div>
      <div class="kpi"><span class="small muted">Línea base</span><b>${r.baseline_cm.toFixed(2)} cm</b><span class="small muted">compará con la regla</span></div>
      <div class="kpi"><span class="small muted">Focal (rectificada)</span><b>${r.focal_px.toFixed(0)} px</b></div>
      <div class="kpi"><span class="small muted">RMS L / R</span><b>${r.rms_left.toFixed(2)} / ${r.rms_right.toFixed(2)}</b></div>
      <div class="kpi"><span class="small muted">Pares · resolución</span><b>${r.n_pairs}</b><span class="small muted">${r.image_size.join("×")}</span></div>
    </div>
    <p class="small muted">Error por par (los rojos superan 1,5× el RMS). Descartá los peores y volvé a calcular.</p>
    <div class="bars">${bars}</div>`;
  el.classList.remove("hidden");
  el.querySelectorAll("[data-del]").forEach((b) =>
    b.addEventListener("click", async () => {
      await removePair(+b.dataset.del);
      b.closest(".bar").remove();
    })
  );
}

// ------------------------------------------------------------- verificar
$("#btnVerify").addEventListener("click", async () => {
  try {
    Calib.verifyPair = await grabPair(0.95);
    const r = await api("/calibration/verify", { method: "POST", body: pairForm(Calib.verifyPair.left, Calib.verifyPair.right) });
    const img = $("#verifyImg");
    img.src = r.image;
    img.dataset.w = r.image_size[0];
    img.dataset.h = r.image_size[1];
    img.classList.remove("hidden");
    $("#verifyZ").textContent = "Clic en la imagen izquierda para medir Z";
  } catch (e) {
    toast(e.message, true);
  }
});

$("#verifyImg").addEventListener("click", async (ev) => {
  if (!Calib.verifyPair) return;
  const img = ev.currentTarget;
  const rect = img.getBoundingClientRect();
  const w = +img.dataset.w, h = +img.dataset.h;
  // La imagen muestra izquierda|derecha lado a lado: 2w de ancho.
  const x = ((ev.clientX - rect.left) / rect.width) * 2 * w;
  const y = ((ev.clientY - rect.top) / rect.height) * h;
  if (x >= w) return toast("Hacé clic sobre la mitad izquierda.");
  $("#verifyZ").textContent = "Midiendo…";
  try {
    const r = await api("/calibration/verify", {
      method: "POST",
      body: pairForm(Calib.verifyPair.left, Calib.verifyPair.right, { x: x.toFixed(1), y: y.toFixed(1) }),
    });
    $("#verifyZ").textContent = r.z_m
      ? `Z = ${(r.z_m * 100).toFixed(1)} cm  (disparidad ${r.disparity_px.toFixed(1)} px)`
      : "Sin disparidad válida en ese punto (zona sin textura u ocluida).";
  } catch (e) {
    toast(e.message, true);
  }
});

// ------------------------------------------------------------- guardar / cargar
$("#btnDownloadCalib").addEventListener("click", async () => {
  try {
    const blob = await api("/calibration?format=yml");
    const a = document.createElement("a");
    a.href = URL.createObjectURL(blob);
    a.download = "stereo_calib.yml";
    a.click();
    URL.revokeObjectURL(a.href);
  } catch (e) {
    toast(e.message, true);
  }
});

$("#uploadCalib").addEventListener("change", async (ev) => {
  const file = ev.target.files[0];
  if (!file) return;
  const fd = new FormData();
  fd.append("file", file);
  try {
    const r = await api("/calibration", { method: "PUT", body: fd });
    await refreshHealth();
    toast(`Calibración cargada: línea base ${r.baseline_cm.toFixed(2)} cm, ${r.image_size.join("×")}.`);
  } catch (e) {
    toast(e.message, true);
  }
  ev.target.value = "";
});

// ------------------------------------------------------------- tablero imprimible
$("#btnBoard").addEventListener("click", () => {
  // Un tablero de N×M esquinas internas tiene (N+1)×(M+1) cuadros. Se dibuja a 300 dpi.
  const cols = +$("#bCols").value + 1, rows = +$("#bRows").value + 1, mm = +$("#bSquare").value;
  const px = Math.round((mm / 25.4) * 300), margin = px;
  const c = document.createElement("canvas");
  c.width = cols * px + 2 * margin;
  c.height = rows * px + 2 * margin;
  const ctx = c.getContext("2d");
  ctx.fillStyle = "#fff";
  ctx.fillRect(0, 0, c.width, c.height);
  ctx.fillStyle = "#000";
  for (let r = 0; r < rows; r++)
    for (let k = 0; k < cols; k++)
      if ((r + k) % 2 === 0) ctx.fillRect(margin + k * px, margin + r * px, px, px);
  ctx.font = `${Math.round(px / 4)}px sans-serif`;
  ctx.fillText(`${cols - 1}x${rows - 1} esquinas internas · cuadro ${mm} mm @300dpi · imprimir al 100%`, margin, c.height - margin / 3);
  const a = document.createElement("a");
  a.href = c.toDataURL("image/png");
  a.download = `tablero_${cols - 1}x${rows - 1}_${mm}mm.png`;
  a.click();
});

drawCoverage();

// Modos "Imagen" y "En vivo", tabla de coordenadas y vista 3D.
const TOMATO_COLOR = "#e0342a";

const fmt = (v, d = 3) => (v === null || v === undefined ? "—" : v.toFixed(d));

function renderTable(el, tomatoes) {
  if (!tomatoes.length) {
    el.innerHTML = `<p class="muted">No se detectaron tomates.</p>`;
    return;
  }
  const rows = tomatoes.map((t) => {
    const p = t.xyz_center_m || [null, null, null];
    return `<tr>
      <td>#${t.id}</td>
      <td>${(t.conf * 100).toFixed(0)}%</td>
      <td>${fmt(p[0])}</td><td>${fmt(p[1])}</td><td>${fmt(t.xyz_mask_m ? t.xyz_mask_m[2] : null)}</td><td>${fmt(t.xyz_center_sgbm_m ? t.xyz_center_sgbm_m[2] : null)}</td><td>${fmt(t.xyz_m ? t.xyz_m[2] : null)}</td><td>${t.metodo_z === "mascaras" ? "máscaras" : (t.metodo_z || "—")}</td>
      <td>${t.diameter_m ? (t.diameter_m * 100).toFixed(1) : "—"}</td>
      <td>${(t.depth_valid_ratio * 100).toFixed(0)}%</td>
    </tr>`;
  }).join("");
  el.innerHTML = `<table>
    <thead><tr><th>ID</th><th>Conf.</th><th>X (m)</th><th>Y (m)</th><th>Z centro máscaras (m)</th><th>Z centro SGBM (m)</th><th>Z sup. SGBM (m)</th><th>Método</th><th>Ø (cm)</th><th>Disp. válida</th></tr></thead>
    <tbody>${rows}</tbody></table>
    <p class="small muted">Z centro máscaras: diferencia entre el centro del tomate en la cámara izquierda y en la derecha (no depende de la textura de la piel). Z centro SGBM: superficie + 1 radio. Z sup. SGBM: punto visible más cercano. "Método" indica cuál se usa para X, Y, el gráfico 3D y el recuadro. Origen en la cámara izquierda: X a la derecha, Y hacia abajo, Z hacia adelante.</p>`;
}

function renderPlot(el, tomatoes) {
  if (typeof Plotly === "undefined") return;
  const pts = tomatoes.filter((t) => t.xyz_center_m);
  const dark = matchMedia("(prefers-color-scheme: dark)").matches;
  const ink = dark ? "#ecebe6" : "#1e1f1c", grid = dark ? "#34352f" : "#e2e0da";
  // Ejes del gráfico: X derecha, Z profundidad, -Y arriba (vista natural).
  const tomatoTrace = {
    type: "scatter3d", mode: "markers+text",
    x: pts.map((t) => t.xyz_center_m[0]),
    y: pts.map((t) => t.xyz_center_m[2]),
    z: pts.map((t) => -t.xyz_center_m[1]),
    text: pts.map((t) => "#" + t.id),
    textposition: "top center",
    hovertemplate: "%{text}<br>X %{x:.3f} m<br>Z %{y:.3f} m<br>Y %{customdata:.3f} m<extra></extra>",
    customdata: pts.map((t) => t.xyz_center_m[1]),
    marker: {
      size: pts.map((t) => Math.max(6, Math.min(30, (t.diameter_m || 0.06) * 250))),
      color: TOMATO_COLOR,
      line: { color: dark ? "#000" : "#fff", width: 1 },
    },
    name: "Tomates",
  };
  const cameraTrace = {
    type: "scatter3d", mode: "markers+text", x: [0], y: [0], z: [0], text: ["cámara"],
    marker: { size: 5, color: ink, symbol: "diamond" }, textposition: "bottom center", hoverinfo: "skip", name: "Cámara",
  };
  const axis = (title) => ({ title, color: ink, gridcolor: grid, zerolinecolor: grid, backgroundcolor: "rgba(0,0,0,0)" });
  Plotly.react(el, [cameraTrace, tomatoTrace], {
    margin: { l: 0, r: 0, t: 0, b: 0 },
    paper_bgcolor: "rgba(0,0,0,0)",
    showlegend: false,
    font: { color: ink },
    uirevision: "keep", // conserva la rotación del usuario entre actualizaciones
    scene: {
      xaxis: axis("X (m)"), yaxis: axis("Z profundidad (m)"), zaxis: axis("−Y altura (m)"),
      aspectmode: "data",
      camera: { eye: { x: 0.2, y: -1.8, z: 0.9 } },
    },
  }, { displaylogo: false, responsive: true });
}

// ============================================================= IMAGEN
const ImageMode = { left: null, right: null };

for (const side of ["Left", "Right"]) {
  $("#file" + side).addEventListener("change", (ev) => {
    ImageMode[side.toLowerCase()] = ev.target.files[0] || null;
    ev.target.parentElement.classList.toggle("has-file", !!ev.target.files[0]);
  });
}

$("#btnSnap").addEventListener("click", async () => {
  try {
    const { left, right } = await grabPair(0.95);
    ImageMode.left = left;
    ImageMode.right = right;
    $("#fileLeft").parentElement.classList.add("has-file");
    $("#fileRight").parentElement.classList.add("has-file");
    toast("Par capturado de las cámaras. Ahora tocá Detectar.");
  } catch (e) {
    toast(e.message, true);
  }
});

$("#btnDetect").addEventListener("click", async () => {
  if (!ImageMode.left || !ImageMode.right) return toast("Elegí las imágenes izquierda y derecha (o capturalas).", true);
  const btn = $("#btnDetect");
  btn.disabled = true;
  $("#imgStatus").textContent = "Procesando…";
  try {
    const r = await api("/detect", {
      method: "POST",
      body: pairForm(ImageMode.left, ImageMode.right, { conf: $("#imgConf").value, images: "true", method: metodo() }),
    });
    $("#imgAnnotated").src = r.annotated_jpg;
    $("#imgDepth").src = r.depth_jpg;
    renderTable($("#imgTable"), r.tomatoes);
    renderPlot($("#imgPlot"), r.tomatoes);
    $("#imgStatus").textContent = `${r.count} tomate(s) · ${r.method === "opencv" ? "OpenCV" : "YOLO"} · ${r.timing_ms.total} ms` +
      (r.proxy_model ? " · modelo sustituto (sin pesos entrenados)" : "");
  } catch (e) {
    $("#imgStatus").textContent = "";
    toast(e.message, true, 6000);
  } finally {
    btn.disabled = false;
  }
});

// ============================================================= EN VIVO
const Live = { ws: null, running: false, last: null, frames: 0, t0: 0, rtt: [] };

function wsUrl() {
  return App.api.replace(/^http/, "ws") + "/ws/stream";
}

function liveError(msg) {
  // Queda escrito junto al botón (el toast desaparece solo y es fácil no verlo).
  const el = $("#liveStats");
  el.textContent = "⚠ " + msg;
  el.style.color = "var(--bad)";
  toast(msg, true, 10000);
  console.error("[en vivo]", msg);
}

async function startLive() {
  if (!camerasOpen() && !(await openCameras())) return;
  $("#liveStats").textContent = "Conectando… (el primer par tarda más: se carga el modelo)";
  $("#liveStats").style.color = "";
  Live.closedByUser = false;
  Live.ws = new WebSocket(wsUrl());
  Live.ws.binaryType = "arraybuffer";
  Live.ws.onopen = () => {
    Live.running = true;
    Live.frames = 0;
    Live.t0 = performance.now();
    $("#btnLive").textContent = "Detener";
    sendParams();
    sendFrame();
  };
  Live.ws.onmessage = (ev) => {
    const msg = JSON.parse(ev.data);
    if (msg.type === "result") {
      Live.last = msg;
      Live.frames++;
      Live.rtt.push(performance.now() - Live.sentAt);
      if (Live.rtt.length > 20) Live.rtt.shift();
      drawOverlay(msg);
      renderTable($("#liveTable"), msg.tomatoes);
      renderPlot($("#livePlot"), msg.tomatoes);
      updateStats(msg);
      sendFrame(); // stop-and-wait: el siguiente par sale recién cuando llega la respuesta
    } else if (msg.type === "error") {
      liveError(msg.detail);
      stopLive();
    }
  };
  Live.ws.onclose = (ev) => {
    if (Live.running && !Live.closedByUser)
      liveError(`Se cortó la conexión con el servidor (código ${ev.code}${ev.reason ? ": " + ev.reason : ""}). ` +
                "Revisá que la celda del servidor siga corriendo en Colab.");
    stopLive();
  };
  Live.ws.onerror = () => console.error("[en vivo] error de WebSocket");
}

function sendParams() {
  if (Live.ws && Live.ws.readyState === 1)
    Live.ws.send(JSON.stringify({ conf: +$("#liveConf").value, method: metodo() }));
}

async function sendFrame() {
  if (!Live.running || !Live.ws || Live.ws.readyState !== 1) return;
  try {
    const q = +$("#liveQuality").value;
    const { left, right } = await grabPair(q);
    const [lb, rb] = await Promise.all([left.arrayBuffer(), right.arrayBuffer()]);
    const msg = new Uint8Array(4 + lb.byteLength + rb.byteLength);
    new DataView(msg.buffer).setUint32(0, lb.byteLength, false);
    msg.set(new Uint8Array(lb), 4);
    msg.set(new Uint8Array(rb), 4 + lb.byteLength);
    Live.sentAt = performance.now();
    Live.ws.send(msg.buffer);
  } catch (e) {
    liveError("No se pudo capturar el par de cámaras: " + e.message);
    stopLive();
  }
}

function stopLive() {
  Live.running = false;
  Live.closedByUser = true;
  if (Live.ws && Live.ws.readyState <= 1) Live.ws.close();
  Live.ws = null;
  $("#btnLive").textContent = "Iniciar";
}

function updateStats(msg) {
  const secs = (performance.now() - Live.t0) / 1000;
  const rtt = Live.rtt.reduce((a, b) => a + b, 0) / Live.rtt.length;
  $("#liveStats").style.color = "";
  $("#liveStats").textContent =
    `${(Live.frames / secs).toFixed(1)} fps · ida/vuelta ${rtt.toFixed(0)} ms · servidor ${msg.timing_ms.total} ms (${msg.method === "opencv" ? "OpenCV" : "YOLO"} ${msg.timing_ms.detect} · SGBM ${msg.timing_ms.disparity}) · ${msg.count} tomate(s)`;
}

function drawOverlay(msg) {
  const video = $("#liveVideo"), canvas = $("#liveOverlay");
  canvas.width = video.videoWidth;
  canvas.height = video.videoHeight;
  const ctx = canvas.getContext("2d");
  ctx.clearRect(0, 0, canvas.width, canvas.height);
  const scale = Math.max(1, canvas.width / 640);
  ctx.lineWidth = 2 * scale;
  ctx.font = `600 ${14 * scale}px system-ui, sans-serif`;
  for (const t of msg.tomatoes) {
    const col = TOMATO_COLOR;
    const poly = t.polygon_raw;
    if (poly.length >= 3) {
      ctx.beginPath();
      poly.forEach(([x, y], i) => (i ? ctx.lineTo(x, y) : ctx.moveTo(x, y)));
      ctx.closePath();
      ctx.fillStyle = col + "55";
      ctx.strokeStyle = col;
      ctx.fill();
      ctx.stroke();
    }
    const [u, v] = t.centroid_raw_px;
    const label = t.xyz_center_m ? `#${t.id}  Z ${t.xyz_center_m[2].toFixed(2)} m` : `#${t.id}  Z ?`;
    const tw = ctx.measureText(label).width;
    ctx.fillStyle = "rgba(0,0,0,0.7)";
    ctx.fillRect(u - tw / 2 - 4 * scale, v - 22 * scale, tw + 8 * scale, 18 * scale);
    ctx.fillStyle = "#fff";
    ctx.fillText(label, u - tw / 2, v - 8 * scale);
  }
}

$("#btnLive").addEventListener("click", () => (Live.running ? stopLive() : startLive()));
$("#liveConf").addEventListener("change", sendParams);

// ============================================================= AJUSTE DE ESCALA
const Escala = { medidas: [] }; // [{zm, zr}] en cm

function factorEscala() {
  const num = Escala.medidas.reduce((a, m) => a + m.zm * m.zr, 0);
  const den = Escala.medidas.reduce((a, m) => a + m.zm * m.zm, 0);
  return den > 0 ? num / den : null;
}

function renderEscala() {
  const k = factorEscala();
  $("#btnEscAplicar").disabled = !k;
  if (!Escala.medidas.length) {
    $("#escTabla").innerHTML = "";
    $("#escInfo").textContent = "";
    return;
  }
  const filas = Escala.medidas.map((m, i) => `<tr><td>${i + 1}</td><td>${m.zm.toFixed(1)}</td><td>${m.zr.toFixed(1)}</td>
    <td>${(m.zm - m.zr).toFixed(1)}</td><td>${(k * m.zm - m.zr).toFixed(1)}</td>
    <td><button class="ghost" data-esc="${i}" title="Quitar">×</button></td></tr>`).join("");
  $("#escTabla").innerHTML = `<table><thead><tr><th>#</th><th>Z medida</th><th>Z real</th><th>Error actual (cm)</th>
    <th>Error con k (cm)</th><th></th></tr></thead><tbody>${filas}</tbody></table>`;
  $("#escTabla").querySelectorAll("[data-esc]").forEach((b) =>
    b.addEventListener("click", () => { Escala.medidas.splice(+b.dataset.esc, 1); renderEscala(); }));
  const B = App.health && App.health.calibration ? App.health.calibration.baseline_cm : null;
  $("#escInfo").textContent = `k = ${k.toFixed(4)}` + (B ? ` → línea base ${B.toFixed(2)} → ${(B * k).toFixed(2)} cm` : "") +
    (Escala.medidas.length < 3 ? " · conviene usar al menos 3 distancias" : "");
}

$("#btnEscUltima").addEventListener("click", () => {
  const t = Live.last && Live.last.tomatoes.find((t) => t.xyz_center_m);
  if (!t) return toast("Primero iniciá el modo En vivo con un tomate a la vista.", true);
  $("#escZm").value = (t.xyz_center_m[2] * 100).toFixed(1);
});

$("#btnEscAgregar").addEventListener("click", () => {
  const zm = +$("#escZm").value, zr = +$("#escZr").value;
  if (!(zm > 0 && zr > 0)) return toast("Completá Z medida y Z real (en cm).", true);
  Escala.medidas.push({ zm, zr });
  $("#escZm").value = "";
  $("#escZr").value = "";
  renderEscala();
});

async function aplicarEscala(body, msg) {
  try {
    const r = await api("/calibration/scale", {
      method: "POST", headers: { "Content-Type": "application/json" }, body: JSON.stringify(body),
    });
    Escala.medidas = []; // las Z anotadas ya no corresponden a la nueva escala
    renderEscala();
    await refreshHealth();
    toast(`${msg}: factor ${r.factor.toFixed(4)}, línea base ahora ${r.baseline_cm.toFixed(2)} cm. Volvé a medir para verificar.`);
  } catch (e) {
    toast(e.message, true, 6000);
  }
}

$("#btnEscAplicar").addEventListener("click", () => aplicarEscala({ factor: factorEscala() }, "Escala corregida"));
$("#btnEscBase").addEventListener("click", () => {
  const b = +$("#escBase").value;
  if (!(b > 0)) return toast("Ingresá la línea base en cm.", true);
  aplicarEscala({ baseline_cm: b }, "Línea base fijada");
});
</script>
</body>
</html>

Writing index.html


## Iniciar el servidor

In [5]:
# ARRANQUE
import urllib.request

import uvicorn

servidor = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
hilo_servidor = threading.Thread(target=servidor.run, daemon=True)
hilo_servidor.start()
for _ in range(60):
    try:
        print(urllib.request.urlopen(f"http://localhost:{PORT}/health").read().decode())
        break
    except Exception:
        time.sleep(0.5)

{"status":"ok","model_loaded":false,"model":null,"model_error":null,"methods":{"modelo":"Modelo YOLO11-seg","opencv":"OpenCV (HSV + watershed)"},"calibrated":true,"calibration":{"image_size":[1280,720],"rms":0.36798735644403574,"baseline_cm":8.138774676806273,"focal_px":1605.931300510672,"created":"2026-11-11T22:53:10"}}


## Publicar con túnel Cloudflare
En Colab imprime la URL pública. En la PC basta con abrir `http://localhost:8000`.

In [6]:
# ARRANQUE
import re

URL = f"http://localhost:{PORT}"
if EN_COLAB:
    tunel_log = open("/content/tunnel.log", "w")
    tunel = subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://localhost:{PORT}"],
                             stdout=tunel_log, stderr=subprocess.STDOUT)
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/tunnel.log").read())
        if m:
            URL = m.group(0)
            break
        time.sleep(1)
print("\n  Abrí la página en →", URL, "\n")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings

  Abrí la página en → https://spouse-double-entitled-uses.trycloudflare.com 



## Estado y detener

In [7]:
# ARRANQUE
import json as _json
print(_json.dumps(health(), indent=2, ensure_ascii=False))
if os.path.isfile(ERRORES_LOG):
    print("\nÚltimos errores del servidor (errores_api.log):")
    print(open(ERRORES_LOG, encoding="utf-8").read()[-4000:])
else:
    print("\nSin errores registrados.")

{
  "status": "ok",
  "model_loaded": false,
  "model": null,
  "model_error": null,
  "methods": {
    "modelo": "Modelo YOLO11-seg",
    "opencv": "OpenCV (HSV + watershed)"
  },
  "calibrated": true,
  "calibration": {
    "image_size": [
      1280,
      720
    ],
    "rms": 0.36798735644403574,
    "baseline_cm": 8.138774676806273,
    "focal_px": 1605.931300510672,
    "created": "2026-11-11T22:53:10"
  }
}

Sin errores registrados.


# DETENER
servidor.should_exit = True
if EN_COLAB:
    tunel.terminate()
print("Servidor detenido.")